In [10]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math


In [11]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [12]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/GSMsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/GSM/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['number_answer'])  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Write to Correct/Incorrect Logs
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            print("wrong")
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:03<10:23,  3.14s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:09<16:46,  5.08s/it]

Accuracy: 2 / 2 = 100.00%


  2%|▏         | 3/200 [00:11<11:59,  3.65s/it]

...

Accuracy: 181 / 198 = 91.41%


100%|█████████▉| 199/200 [09:13<00:03,  3.46s/it]

Accuracy: 182 / 199 = 91.46%


100%|██████████| 200/200 [09:17<00:00,  2.79s/it]

Accuracy: 183 / 200 = 91.50%


In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/GSM/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['number_answer'])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:01<06:06,  1.84s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:05<09:24,  2.85s/it]

Accuracy: 2 / 2 = 100.00%


  2%|▏         | 3/200 [00:06<07:25,  2.26s/it]

...

Accuracy: 177 / 198 = 89.39%


100%|█████████▉| 199/200 [06:37<00:03,  3.01s/it]

Accuracy: 178 / 199 = 89.45%


100%|██████████| 200/200 [06:38<00:00,  1.99s/it]

Accuracy: 179 / 200 = 89.50%


In [14]:
import concurrent.futures
import math
import re
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/GSM/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Processing Function ===
def process_data(d):
    global acc, total
    q = d['question']
    a = float(d['number_answer'])  # Ground truth answer

    # === Prompt Setup for Complex CoT ===
    prompt_q = (
        CCoT_prompt_examples +
        "\nQ: " + q + "\n\n"
        "Please reason through this problem using a complex, multi-step chain of thought:\n"
        "Step 1: Clearly state all given information and any assumptions.\n"
        "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
        "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
        "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
        "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
        "Step 6: Double-check the solution for errors or unreasonable results.\n"
        "Finish your response with: the answer is <answer>"
    )

    messages = [
        {
            "role": "system",
            "content": (
                "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
            )
        },
        {"role": "user", "content": prompt_q}
    ]

    # === Get Response ===
    response = completion_with_backoff(messages)
    ans_model = response.choices[0].message.content.strip()

    # === Improved Answer Extraction ===
    match = re.search(
        r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
        ans_model,
        re.IGNORECASE
    )
    if match:
        extracted_raw = match.group(1).strip()
        extracted = clean_and_truncate(extracted_raw)
    else:
        extracted = None

    # === Log Block
    log_block = (
        f'Q: {q}\n'
        f'A_model:\n{ans_model}\n'
        f'Extracted:\n{extracted}\n'
        f'A:\n{a}\n\n'
    )

    if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
        acc += 1
        return log_block, True
    else:
        return "❌ Incorrect or Invalid\n" + log_block, False

# === Main Loop with Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_data, d) for d in dev_data]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(dev_data)):
            log_block, is_correct = future.result()
            if is_correct:
                fd.write(log_block)
            else:
                bad_fd.write(log_block)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:06<21:56,  6.62s/it]

Accuracy: 1 / 1 = 100.00%


  2%|▏         | 3/200 [00:07<06:14,  1.90s/it]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/200 [00:09<06:05,  1.86s/it]

...

Accuracy: 153 / 198 = 77.27%


100%|█████████▉| 199/200 [10:19<00:02,  2.00s/it]

Accuracy: 153 / 199 = 76.88%


100%|██████████| 200/200 [14:15<00:00,  4.28s/it]

Accuracy: 154 / 200 = 77.00%
